# Supp Table 7 — architecture ablation (per-sample)

**🔴 heavy (GPU / multi-GB / long)** · source: `notebooks/ablation_primary_models.py`

🔴 Same heavy training run as Supp Table 6.

## Configuration — edit the paths, then run

In [ ]:
import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")
try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print("[note] scanpy/anndata not available:", e)

# ── EDIT THESE PATHS to match your environment ──
REPO_ROOT     = Path("/path/to/spatnic")          # this repository
BACKUP_ROOT   = Path("/path/to/backup")           # integrate_adata_filtered.h5ad, galaxy scores, Liver meta
DATA_ROOT     = Path("/path/to/data")             # GxD concat, lung annotated, c2l refs, spatnic_models, GxD_Xenium
BENCHMARK_DB  = Path("/path/to/benchmark_db")     # Xenium/VisiumHD/MERFISH/CosMx + adata_hvg_*
VISIUMHD_ROOT = Path("/path/to/VisiumHD")         # Visium HD ADC track
WEIGHTS_DIR   = Path.home() / ".spatnic" / "weights"

# ── Derived ──
NB     = REPO_ROOT / "notebooks"
COMP   = NB / "comparison_results"
VHD    = VISIUMHD_ROOT
MODELS = DATA_ROOT / "spatnic_models"
PAPER  = REPO_ROOT / "paper"
BASE_DIR  = VHD          # VisiumHD ADC notebook global
THRESHOLD = 0.9          # overridden to 0.5 by the Fig 5 shortcut setup cell
sys.path[:0] = [str(REPO_ROOT / "scripts"), str(NB)]
if NB.exists():
    os.chdir(NB)         # extracted cells were written for cwd = notebooks/

def _tbl(csv, n=None):
    p = Path(csv)
    if not p.exists():
        print("[missing]", p); return None
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p)
    display(df.head(n) if n else df); return df

def _run(script, show=None, n=None):
    import subprocess
    cmd = f"python notebooks/{script}"
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT), capture_output=True, text=True)
    print((r.stdout or "")[-3000:])
    if r.returncode: print("STDERR:\n", (r.stderr or "")[-2000:])
    if show: _tbl(COMP / show, n)


## Regenerate (runs the real metric program)

In [ ]:
# _run("ablation_primary_models.py")   # 🔴 uncomment to retrain (GPU, long-running)
_tbl(COMP / "ablation_primary/metrics_persample.csv")

## Result (current cached values)

**Ablation per-sample** (`metrics_persample.csv`, 12 rows)

| test_set | method | n_samples | auc_pr_mean | auc_pr_std | auc_roc_mean | auc_roc_std | f1_mean | f1_std | mcc_mean | mcc_std | accuracy_mean | accuracy_std | balanced_acc_mean | balanced_acc_std |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| internal | Student-t VAE (SPATNIC) | 37 | 0.9237242309337172 | 0.1544040932279963 | 0.868218827451744 | 0.1531405475195521 | 0.9032437057495526 | 0.1439020306220322 | 0.5807569580685645 | 0.3493996234366083 | 0.9052811862663114 | 0.1255983450428488 | 0.8086265290582116 | 0.1778424261957753 |
| internal | Gaussian VAE | 37 | 0.9021480969032868 | 0.1621056724799913 | 0.8321010102070799 | 0.1568294332525735 | 0.8551364796347553 | 0.1491487601831849 | 0.4948592467832762 | 0.3169230579857741 | 0.8602649765342467 | 0.1150071578054421 | 0.778671330435822 | 0.1579277996655315 |
| internal | MLP (supervised) | 37 | 0.9034363140389928 | 0.1611516870168624 | 0.8548164627825515 | 0.136042303958132 | 0.8823007823116058 | 0.1475987045174005 | 0.5403365345408172 | 0.3136590549779408 | 0.8897201014785847 | 0.1088158111388857 | 0.8058785486630288 | 0.1480684948349354 |
| internal | Random Forest | 37 | 0.9032474595200662 | 0.163553223904619 | 0.8368161343125786 | 0.1489367096508865 | 0.8328072297308011 | 0.1999471527122535 | 0.3280995257311464 | 0.3361436903354765 | 0.8262627752630749 | 0.1822354985416413 | 0.6582395120643987 | 0.171659182109075 |
| internal | Logistic Regression | 37 | 0.9079401510967864 | 0.1577074773781827 | 0.8440665065513295 | 0.1625651346487875 | 0.876399232053225 | 0.1476701210571692 | 0.5315894161680224 | 0.3270313534376647 | 0.8828287690849842 | 0.105614522630997 | 0.7936153442847976 | 0.1570123319362899 |
| internal | XGBoost | 37 | 0.9004041934075546 | 0.1695009410896171 | 0.8240893649757913 | 0.1578040201672797 | 0.8509378728379595 | 0.1622531898841063 | 0.4514540870512584 | 0.336895293059358 | 0.8550104653087637 | 0.1260929062462935 | 0.7420811025636885 | 0.1618649072799744 |
| external | Student-t VAE (SPATNIC) | 7 | 0.9879051451785268 | 0.0224758476239994 | 0.9874805924974512 | 0.0152539984018204 | 0.9516690635413424 | 0.0464797410783667 | 0.8171054910557862 | 0.0883449666961085 | 0.9405013264395566 | 0.0385802698080626 | 0.952062259261746 | 0.038750661788658 |
| external | Gaussian VAE | 7 | 0.950069421885014 | 0.0835798780347158 | 0.9410399614928112 | 0.0506525390454123 | 0.8391051400893483 | 0.0785501514110919 | 0.5348757361907561 | 0.135981558578387 | 0.7896444027655131 | 0.0829120267012076 | 0.8468154259816331 | 0.059305520834625 |
| external | MLP (supervised) | 7 | 0.920728148226934 | 0.191667365725627 | 0.9320686636515166 | 0.1424696410043603 | 0.9005093142923675 | 0.111659660000035 | 0.6640168028923478 | 0.1880233035657705 | 0.864674518182302 | 0.1308726348207344 | 0.8921435060489696 | 0.1219157805661374 |
| external | Random Forest | 7 | 0.9675542081623544 | 0.0355109687878456 | 0.9495126215159596 | 0.0308355876700553 | 0.9046724994875908 | 0.1260056418456304 | 0.6962431018622887 | 0.1978228106435734 | 0.8681526182246405 | 0.1608382081651888 | 0.8291953359821417 | 0.1236834105275934 |
| external | Logistic Regression | 7 | 0.9239765119007636 | 0.1806833422259955 | 0.9468498561939436 | 0.1105911785483646 | 0.9062391756495288 | 0.1061863581756583 | 0.7115318980549464 | 0.1599581511257508 | 0.8858289060188614 | 0.1072281596444917 | 0.9055620089765244 | 0.1034333261937092 |
| external | XGBoost | 7 | 0.9477964205534752 | 0.1056966013666608 | 0.9465721039970886 | 0.0798338479769801 | 0.8454995843455997 | 0.1014555701307945 | 0.5565236090764524 | 0.157741953519625 | 0.8033353175987793 | 0.0995509894274577 | 0.852026124306555 | 0.0918155027806775 |